## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    TypeVar,
    Union,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log, ceil
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline
from scipy.stats import linregress
from scipy.signal import savgol_filter, find_peaks, peak_prominences, peak_widths
from pint import Quantity
import bottleneck

from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing import helpers
from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    # NonReactorDataframeColumn,
    # SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.processing.neutron_window_strategy.strategy_factory \
    import NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy \
    import AbstractNeutronStrategy
# from data_processing.helpers import (
#     # get_input_with_default,
#     # get_input_required,
#     # input_experiment_ids,
#     stop,
#     get_midpoints_from_bins
# )


### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> proc_types.NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = proc_types.NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
# def relative_rmse(x: pd.Series | float, x_err: pd.Series | float, y: pd.Series | float, y_err: pd.Series | float) -> pd.Series | float:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    # rel_sq_x = relative_square_error(x, x_err)
    # rel_sq_y = relative_square_error(y, y_err)
    # rel_sq_sum = rel_sq_x + rel_sq_y
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 40,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pd.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    
    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)
    
    signals_np = -signals_np + baselines + offset
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_col: str = DetectorDataframeColumn.ENERGY.value,
    psd_col: str = DetectorDataframeColumn.PSD.value,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = df[adc_col]
    y = df[psd_col]

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

In [ ]:
def integrate_signals(y: np.array, x: Union[np.array, None] = None) -> float:
    if x is None:
        _, x = np.mgrid[:y.shape[0], :y.shape[1]]
    # TODO test that x and y are 2D
    if len(y.shape) != 2:
        raise ValueError("y array must be 2 dimensional")
    # if x.shape != y.shape:
    #     raise ValueError("x and y must have the same shape")

    left_ys = y[:, :-1]
    right_ys = y[:, 1:]
    left_xs = x[..., :-1]
    right_xs = x[..., 1:]

    # trapezoidal integration
    # area between points has shape of trapezoid
    # trap. area = rectangle where height is mean of parallel edge heights
    trap_mean_heights = (left_ys + right_ys) / 2
    trap_widths = right_xs - left_xs
    trap_areas = trap_mean_heights * trap_widths
    integral = np.sum(trap_areas, axis=1)
    return integral

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))


def simple_deriv(arr):
    hi = arr[2:]
    lo = arr[:-2]
    delta = hi - lo
    prefix = np.empty((1,))
    suffix = np.empty((1,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, delta, suffix))

In [ ]:
def integrate_pulses(pulse_data: np.array, t_start: int, t_end: int):
    left_vals = pulse_data[:, t_start:t_end]
    right_vals = pulse_data[:, t_start+1:t_end+1]
    # left_vals = pulse_series.loc[t_start:t_end]
    # right_vals = pulse_series.loc[t_start+1:t_end+1]

    # print(left_vals.shape, right_vals.shape)
    # print(left_vals)
    # print(right_vals)
    midpoints = (left_vals + right_vals) / 2
    # print(midpoints)
    column_areas = midpoints * 2  # 2 ns between data points
    # print(column_areas)
    areas = column_areas.sum(axis=1)
    return areas


def integrate_pulse_gates(
    pulses: pd.DataFrame, t1: int, t2: int, t3: int, baseline_count: int = 40
) -> pd.DataFrame:
    baseline_adjust = 0
    # baseline_window = 25

    if not pd.api.types.is_numeric_dtype(pulses.values):
        raise ValueError("DataFrame values must all be numeric type")
    if pulses.shape[1] != 200:
        raise ValueError("DataFrame rows must be 200 samples long")
    if not all([0 <= x <= 398 for x in [t1, t2, t3]]):
        raise ValueError("Times must all be between 0 and 398 (inclusive)")
    if not (t2 > t1):
        raise ValueError("t2 must be greater than t1")
    if not (t3 > t2):
        raise ValueError("t3 must be greater than t2")

    idx_1 = ceil(t1 / 2)
    idx_2 = ceil(t2 / 2)
    idx_3 = ceil(t3 / 2)

    pulses_np = pulses.to_numpy()
    # baselines = np.trunc(
    #     pulses_np[:, :baseline_count].mean(axis=1).reshape(-1, 1)
    # )
    # baselines = baselines + baseline_adjust
    # long_slices = pulses_np[:, idx_1:idx_3]
    # short_slices = pulses_np[:, idx_1:idx_2]
    # peak_slices = pulses_np[:, 30:60]

    # q_long = (baselines - long_slices).sum(axis=1)
    # q_short = (baselines - short_slices).sum(axis=1)
    q_long = integrate_pulses(pulses_np, idx_1, idx_3)
    q_short = integrate_pulses(pulses_np, idx_1, idx_2)
    # peak_heights = (baselines - peak_slices).max(axis=1)

    psd_df = pd.DataFrame(
        {"Q_LONG": q_long, "Q_SHORT": q_short},
        index=pulses.index
    )
    return psd_df

In [ ]:
def two_point_inv_lerp(y: float, p1: tuple[float, float], p2: tuple[float, float]) -> float:
    deltas = tuple([n2 - n1 for n1, n2 in zip(p1, p2)])
    m = deltas[1] / deltas[0]
    x1, y1 = p1
    if m == 0:
        return x1
    x = (y - y1) / m + x1
    return x


def calculate_q_fom_critical(
    fit_df: pd.DataFrame,
    # q_limits: tuple[float, float] | None = None
    # q_limit_hi: float | None = None
) -> tuple[float | None, float | None]:
    fom_crit = 1.27
    # if q_limits is None:
    #     q_limits = (2500, 60000)  # x axis area with clean FOM curve
    # if q_limit_hi is None:
    #     q_limit_hi = 60000
    fom_data = fit_df["fom"]
    slice_energy_min = fit_df["slice_energy_min"]
    slice_energy_max = fit_df["slice_energy_max"]
    slice_energy_mid = (slice_energy_min + slice_energy_max) / 2
    fom_x = slice_energy_mid.values
    fom_y = fom_data.values
    delta_y = fom_y[2:] - fom_y[:-2]
    stable_end_idx = np.argmax(fom_x >= 20000)

    # use moving average of delta_y to find stable region (delta_y <= threshold)
    # find first cross in stable region
    window = 5
    # bottleneck window functions use look-behind windows and fill missing with nan
    # so first window-1 values are always nan; we need to convert to look-ahead
    # 
    delta_y_mov_max = bottleneck.move_max(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    delta_y_mov_min = bottleneck.move_min(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    stable_max_delta = delta_y_mov_max < 0.25
    stable_min_delta = delta_y_mov_min > -0.25
    stable_delta = (stable_max_delta & stable_min_delta)
    if not stable_delta.any():
        return None, None
    stable_start_idx = np.argmax(stable_delta)
    stable_start_x = fom_x[stable_start_idx]
    cross_search_slice = fom_y[stable_start_idx:stable_end_idx]
    if not (cross_search_slice >= fom_crit).any():
        return None, stable_start_x
    # argmax gets i in slice, need to add slice start to get i in original list
    fom_critical_idx = np.argmax(cross_search_slice >= fom_crit) + stable_start_idx
    
    # possible_crosses = np.where((fom_y[1:] >= 1.27) & (fom_y[:-1] <= 1.27))[0]
    # # dd_sums = []
    # fom_critical_idx = None
    # for possible_cross in possible_crosses:
    #     pass  # STUB
    # slice_lo = possible_cross - 3 if possible_cross - 3 >= 0 else 0
    # slice_hi = possible_cross + 2
    # deltas = delta_y[slice_lo:slice_hi]
    # delta_deltas = deltas[1:] - deltas[:-1]
    # dd_sum = abs(delta_deltas).sum()
    # dd_sums.append(dd_sum)
    
    # idx_best_cross = np.argmin(np.nan_to_num(dd_sums, nan=np.inf))
    # fom_critical_idx = possible_crosses[idx_best_cross] + 1
    if fom_critical_idx is None:
        q_fom_critical = None
    elif fom_critical_idx > 0:
        # x_crit_bounds = tuple([fom_x[fom_critical_idx+x] for x in [-1, 0]])
        # y_crit_bounds = tuple([fom_y[fom_critical_idx+x] for x in [-1, 0]])
        p1 = fom_x[fom_critical_idx-1], fom_y[fom_critical_idx-1]
        p2 = fom_x[fom_critical_idx], fom_y[fom_critical_idx]
        if p1[1] > fom_crit:  # we can't make lerp extrapolate!
            q_fom_critical = None
        else:
            q_fom_critical = two_point_inv_lerp(fom_crit, p1, p2)
    else:
        q_fom_critical = fom_x[fom_critical_idx]
    return q_fom_critical, stable_start_x

In [ ]:
def calculate_q_fom_critical_from_pulses(
    times: np.ndarray, pulses: pd.DataFrame
) -> tuple[float | None, float | None]:
    t1, t2, t3, *_ = times.flatten()
    psd_df = integrate_pulses(pulses, t1, t2, t3)
    psd_df = calculate_psd(psd_df)
    histogram, energy_bin_edges, psd_bin_edges = get_psd_energy_histogram(psd_df)
    fit_df, _ = scan_histogram_slices(histogram, energy_bin_edges, psd_bin_edges)
    q_fom_critical = calculate_q_fom_critical(fit_df)
    return q_fom_critical


def generate_search_grid(
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
) -> np.ndarray:
    if isinstance(counts, tuple):
        t1_counts, t2_counts, t3_counts = counts
    else:
        t1_counts, t2_counts, t3_counts = counts, counts, counts
    t1_range = np.linspace(*t1_limits, num=t1_counts)
    t2_range = np.linspace(*t2_limits, num=t2_counts)
    t3_range = np.linspace(*t3_limits, num=t3_counts)
    t_grid = np.meshgrid(t1_range, t2_range, t3_range)
    t_grid = tuple([np.ravel(grid_element) for grid_element in t_grid])
    t_grid_stacked = np.vstack(t_grid)  # shape (3, n)
    return t_grid_stacked


T = TypeVar("T")


def search_grid(
    grid_search_fn: Callable[[np.ndarray, pd.DataFrame], T],
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
    pulses: pd.DataFrame,
    cores: int = 4,
    use_chunks: bool = False,
) -> tuple[np.ndarray, list[T]]:
    t_grid_stacked = generate_search_grid(
        t1_limits, t2_limits, t3_limits, counts
    )

    pool_size = max(
        2 * cores, 4
    )
    # based on https://jupyter-tutorial.readthedocs.io/en/stable
    # /performance/multiprocessing.html
    if use_chunks:
        chunksize, extra = divmod(len(t_grid_stacked.shape[1]), pool_size * 4)
        if extra > 0:
            chunksize += 1
    else:
        chunksize = "auto"

    parallelizer = Parallel(
        n_jobs=pool_size, batch_size=chunksize, max_nbytes=1e6, verbose=10
    )
    loop_result = parallelizer(
        delayed(grid_search_fn)(timesarray, pulses)
        for timesarray in t_grid_stacked.T
    )
    return t_grid_stacked, loop_result


def grid_search_fom(
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
    pulses: pd.DataFrame,
    cores: int = 4,
    use_chunks: bool = False,
) -> np.ndarray:
    t_grid_stacked, fom_values = search_grid(
        calculate_q_fom_critical_from_pulses,
        t1_limits,
        t2_limits,
        t3_limits,
        counts,
        pulses,
        cores,
        use_chunks
    )
    
    threshold_energies, threshold_search_starts = list(zip(*fom_values))
    threshold_energies = np.array(threshold_energies, dtype=float)
    threshold_search_starts = np.array(threshold_search_starts, dtype=float)

    return np.vstack(
        [t_grid_stacked, threshold_energies, threshold_search_starts],
        dtype=float
    ).T  # shape (n, 5)

In [ ]:
def calculate_plot_grid_dimensions(n: int, max_cols: int = 4) -> tuple[int, int]:
    ncols = min(n, max_cols)
    nrows = ceil(n / ncols)
    return (nrows, ncols)

In [ ]:
def display_plot_grid(
    grid_plot_fn: Callable[[mpl.axes.Axes, T], None],
    grid_plot_data: list[T],
    grid_count: int,
    max_cols: int
) -> tuple[mpl.figure.Figure, np.ndarray[mpl.axes.Axes]]:
    nrows, ncols = calculate_plot_grid_dimensions(grid_count, max_cols=max_cols)
    fig, axs = plt.subplots(
        nrows, ncols, figsize=(8*ncols, 8*nrows)
    )
    axs = axs.flatten()
    for ax, plot_data in zip(axs, grid_plot_data):
        grid_plot_fn(ax, plot_data)
    return fig, axs

In [ ]:
def pretty_format_duration(duration: float) -> str:
    out_seconds = duration % 60
    dur_minutes = int(duration / 60)
    if dur_minutes == 0:
        return f"{out_seconds:.1f} s"
    out_minutes = dur_minutes % 60
    dur_hours = int(dur_minutes / 60)
    if dur_hours == 0:
        return f"{out_minutes} m {out_seconds:.1f} s"
    else:
        return f"{dur_hours} h {out_minutes} m {out_seconds:.1f} s"

## Data Loading

### Loading Params

In [ ]:
isotope_datasets = {
    # 6: "241AmBe",
    # 7: "241AmBe",
    # 8: "241AmBe",
    22: "22Na",
    # 23: "22Na",
    # 24: "22Na",
    35: "60Co",
    # 36: "60Co",
    # 37: "60Co",
    # 38: "60Co",
    72: "137Cs",
    # 73: "137Cs",
    90: "152Eu",
    97: "40K"
}
isotope_datasets = {f"ID-383.{k}": v for k, v in isotope_datasets.items()}

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in isotope_datasets.keys()
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
# detector_code = helpers.get_input_required(
#     """\
# Which detector was used?
# 1: Original detector (detector 1)
# 2: New detector (detector 2)
# """,
#     [Detector.ZERO, Detector.ONE],
#     lambda x: Detector(int(x)-1)
# )
detector_code = proc.Detector.ZERO

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
# fit_input = helpers.get_input_with_default(
#     """\
# Which bimodal fit type do you want to use?
# 1: Bounds based
# 2: Peak finder based (default)
# Press Enter for default
# """,
#     default_fit_input,
#     int
# )
fit_input = 2

fit_styles: dict[int, proc_types.SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
# bin_length = helpers.get_input_with_default(
#     "Enter bin length (in seconds), or press Enter for default (300 s)",
#     300,
#     int
# )
bin_length = 30
bin_string = f"{bin_length}S"

In [ ]:
analysis_timestamp = datetime.now().strftime("%Y-%m%b-%d-%H-%M-%S")
overall_settings = {
    'calibration_type': repr(calib_key),
    'fitting_style': fit_style,
    'window_settings': repr(settings),
    'bin_length': bin_length
}

In [ ]:
analysis_timestamp

### Loading and Initial Processing

In [ ]:
figure_data = {k: {} for k in ["a", "b", "c"]}

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id)
    exp_data["raw_signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Ensure that index matches between signals and CAEN data
for exp_id, exp_data in experiment_neutron_data.items():
    # print(exp_id)
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    # print(unclassified_df)
    signals_df = exp_data["raw_signals_df"]
    # print(signals_df)

    unclassified_index: pd.Index = unclassified_df.index
    signals_index: pd.Index = signals_df.index
    clean_index = unclassified_index.intersection(signals_index)

    unclassified_df = unclassified_df.loc[clean_index]
    signals_df = signals_df.loc[clean_index]

    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df
    exp_data["raw_signals_df"] = signals_df

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Data Processing

### Neutron Classification

In [ ]:
# Generate histogram
start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

### Pulse Integral Distribution

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    # neutrons_only = psd_report.query(n_class_col_name).copy()
    # exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    signals_df = exp_data["raw_signals_df"]
    g_signals_df = signals_df.loc[gamma_only.index].astype("int32")
    g_signals_df = correct_raw_signals(g_signals_df)
    
    # gamma_only["peak_height"] = g_signals_df.max(axis=1)
    # print(gamma_only["peak_height"].max())
    # print(gamma_only["peak_height"].min())
    g_signals_np = g_signals_df.to_numpy()
    g_signals_time = g_signals_df.columns.map(int).to_numpy() * 2
    g_integrals = integrate_signals(g_signals_np, g_signals_time)
    
    exp_data["g_signals_df"] = g_signals_df
    gamma_only["integral"] = g_integrals
    # print(gamma_only.head())
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    # peak_height = gamma_only["peak_height"]
    integral = gamma_only["integral"]
    
    bin_start = 0
    bin_end = 310000
    bin_size = 1000
    bins = np.arange(bin_start, bin_end+bin_size, step=bin_size)
    
    Z, *_ = np.histogram(integral, bins=bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "gamma": {"standard": Z, "bins": bins},
    }

In [ ]:
# moving average
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_g_histogram = phd_histogram_data["gamma"]["standard"]
    # phd_g_histogram = phd_histogram_data["gamma"]["standard"]

    window_length = 37 if isotope_datasets[exp_id] == "60Co" else 17
    phd_g_moving_average = moving_average_centered(phd_g_histogram, n=window_length)
    # phd_g_moving_average = moving_average_centered(phd_g_histogram)

    phd_histogram_data["gamma"]["moving_average"] = phd_g_moving_average
    # phd_histogram_data["gamma"]["moving_average"] = phd_g_moving_average
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

### Derivative

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_g_histogram = phd_histogram_data["gamma"]["moving_average"]

    phd_deriv = simple_deriv(phd_g_histogram)
    phd_histogram_data["gamma"]["derivative"] = phd_deriv

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_deriv = phd_histogram_data["gamma"]["derivative"]

    phd_deriv_filtered = savgol_filter(phd_deriv, 27, 3)
    phd_histogram_data["gamma"]["filter_deriv"] = phd_deriv_filtered

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
    phd_deriv_filtered = phd_histogram_data["filter_deriv"]

    isotope_name = isotope_datasets[exp_id]
    print(isotope_name)
    match isotope_name:
        case "241AmBe":
            pf_params = {
            #     "height": (9, 12.5),
            #     "prominence": 4.5,
                "width": 20,
                "prominence": 20
            }
        case "22Na":
            pf_params = {
                # "width": 20,
                # "prominence": 100
                "prominence": 250
            }
        case "60Co":
            pf_params = {
                # "prominence": 120
                "prominence": 700
            }
        case "137Cs":
            pf_params = {
                # "width": 20,
                # "prominence": 200
                "prominence": 4,
                "height": (100, 1250)
            }
        case "152Eu":
            pf_params = {
                # "width": 10,
                # "prominence": 5
                "prominence": 9
            }
        case "40K":
            pf_params = {
                "prominence": 100
            }
        case _:
            pf_params = {}
    
    phd_deriv_filtered = -1 * phd_deriv_filtered
    peak_idx, *rest = find_peaks(phd_deriv_filtered, **pf_params)
    print(*rest)
    prominences = peak_prominences(phd_deriv_filtered, peak_idx)
    widths = peak_widths(phd_deriv_filtered, peak_idx)
    # print(prominences)
    phd_histogram_data["peak_idx"] = peak_idx
    phd_histogram_data["peak_prominences"] = prominences
    phd_histogram_data["peak_widths"] = widths

### Compton Edge

In [ ]:
isotope_compton_edges = {
    # "241AmBe": [4.4],  # 4.9?
    "22Na": [0.341, 1.062],
    "60Co": [0.960],  # 1.116 not resolved
    "137Cs": [0.478],
    "152Eu": [0.197, 0.587, 0.904, 1.192],  # 0.039 not found
    "40K": [1.243]
}
compton_locs = {
    isotope_code: {
        compton_energy: [] for compton_energy in compton_energies
    }
    for isotope_code, compton_energies in isotope_compton_edges.items()
}

In [ ]:
# Calculate Compton edges
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
    
    bins = phd_histogram_data["bins"]
    peak_idxs = phd_histogram_data["peak_idx"]
    isotope_code = isotope_datasets[exp_id]
    isotope_compton_locs = compton_locs[isotope_code]

    bin_mids = (bins[1:] + bins[:-1]) / 2
    peak_integrals = bin_mids[peak_idxs]
    compton_energies = sorted(list(isotope_compton_locs.keys()))

    for integ_val, compton_energy in zip(peak_integrals, compton_energies):
        integrals_list = isotope_compton_locs[compton_energy]
        integrals_list.append(integ_val)
print(compton_locs)

### Calibration Curve

In [ ]:
calibration_data = []
for isotope_code, compton_edge_data in compton_locs.items():
    for comp_edge_energy, integrals_list in compton_edge_data.items():
        mean_integral = np.mean(integrals_list)
        # print(isotope_code, comp_edge_energy, mean_integral)
        calibration_data.append((comp_edge_energy, mean_integral, isotope_code))
print(calibration_data)

In [ ]:
# def calibration_curve(L, a, b, c):
#     return a * np.log10(b * L + 1) + c * L

def calibration_curve(L, a, b):
    return a * L + b

In [ ]:
L_list, integ_list, isotope_list = zip(*calibration_data)

In [ ]:
print(L_list)

In [ ]:
fit_params, fit_cov = curve_fit(
    calibration_curve,
    L_list,
    integ_list,
    # p0=(2500, 2000, 50000),
    maxfev=80000,
    # bounds=(
    #     (-np.inf, 0, 0),
    #     (np.inf, np.inf, np.inf)
    # )
)
print(fit_params)
print(np.sqrt(np.diag(fit_cov)))

### Subplot Processing

#### Figure 17a

In [ ]:
isotope_names = ["60Co", "137Cs", "40K"]
plot_data = figure_data["a"]

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    isotope_name = isotope_datasets[exp_id]
    if isotope_name in isotope_names:
        phd_histogram_data = exp_data[
            ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
        energy_bins = phd_histogram_data["bins"]
        phd_histogram = phd_histogram_data["moving_average"]
        phd_deriv_filt = phd_histogram_data["filter_deriv"]
        peak_idx = phd_histogram_data["peak_idx"]
        # peak_prominences = phd_histogram_data["peak_prominences"]
        # peak_widths = phd_histogram_data["peak_widths"]
        # print(isotope_name)
        # print(peak_prominences)
        # print(peak_widths)

        energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
        phd_histogram = np.nan_to_num(phd_histogram)
        phd_deriv_filt = np.nan_to_num(phd_deriv_filt)
        peak_x = energy_bin_mids[peak_idx]
        peak_y = phd_histogram[peak_idx]
        deriv_y = phd_deriv_filt[peak_idx]

        isotope_plot_data = {
            "bins": energy_bins,
            "counts": phd_histogram,
            "derivative": phd_deriv_filt,
            "peak_coords": (peak_x, peak_y),
            "deriv_coords": (peak_x, deriv_y)
        }
        plot_data[isotope_name] = isotope_plot_data

### Figure 17b pt 1

In [ ]:
plot_data = figure_data["b"]

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    isotope_name = isotope_datasets[exp_id]
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
    energy_bins = phd_histogram_data["bins"]
    phd_histogram = phd_histogram_data["moving_average"]
    phd_deriv_filt = phd_histogram_data["filter_deriv"]
    peak_idx = phd_histogram_data["peak_idx"]
    compton_edges = isotope_compton_edges[isotope_name]
    # peak_prominences = phd_histogram_data["peak_prominences"]
    # peak_widths = phd_histogram_data["peak_widths"]
    # print(isotope_name)
    # print(peak_prominences)
    # print(peak_widths)

    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
    phd_histogram = np.nan_to_num(phd_histogram)
    phd_deriv_filt = np.nan_to_num(phd_deriv_filt)
    peak_x = energy_bin_mids[peak_idx]
    peak_y = phd_histogram[peak_idx]
    # deriv_y = phd_deriv_filt[peak_idx]

    isotope_plot_data = {
        "bins": energy_bins,
        "counts": phd_histogram,
        "derivative": phd_deriv_filt,
        "peak_coords": (peak_x, peak_y),
        "compton_edges": compton_edges
        # "deriv_coords": (peak_x, deriv_y)
    }
    plot_data[isotope_name] = isotope_plot_data

### Figure 17b pt 2

In [ ]:
plot_data = figure_data["c"]

In [ ]:
calib_curve_data = []
for exp_id, exp_data in experiment_neutron_data.items():
    isotope_name = isotope_datasets[exp_id]
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["gamma"]
    energy_bins = phd_histogram_data["bins"]
    # phd_histogram = phd_histogram_data["moving_average"]
    # phd_deriv_filt = phd_histogram_data["filter_deriv"]
    peak_idx = phd_histogram_data["peak_idx"]
    compton_edges = isotope_compton_edges[isotope_name]

    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
    # phd_histogram = np.nan_to_num(phd_histogram)
    # phd_deriv_filt = np.nan_to_num(phd_deriv_filt)
    peak_x = energy_bin_mids[peak_idx]
    # peak_y = phd_histogram[peak_idx]
    # deriv_y = phd_deriv_filt[peak_idx]

    for edge_adc, edge_l in zip(peak_x, compton_edges):
        calib_curve_data.append((isotope_name, edge_adc, edge_l))
    plot_data["calib_curve_data"] = calib_curve_data

In [ ]:
calib_curve_data = plot_data["calib_curve_data"]
_, calib_adc, calib_l = zip(*calib_curve_data)
fit_result = linregress(calib_l, calib_adc)
print(fit_result.slope, fit_result.stderr)
print(fit_result.intercept, fit_result.intercept_stderr)
print(fit_result.rvalue, np.square(fit_result.rvalue))
plot_data["fit_result"] = fit_result

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"
transparent = "#00000000"

### Plot Functions

In [ ]:
def plot_figure_17a(
    ax: mpl.axes.Axes,
    isotope: str
):
    plot_data = figure_data["a"][isotope]

    bins = plot_data["bins"]
    counts = plot_data["counts"]
    deriv = plot_data["derivative"]
    peak_coords = plot_data["peak_coords"]
    deriv_coords = plot_data["deriv_coords"]

    peak_coords = list(zip(*peak_coords))
    deriv_coords = list(zip(*deriv_coords))
    bin_mids = (bins[1:] + bins[:-1]) / 2

    divider = make_axes_locatable(ax)
    ax_d = divider.append_axes("bottom", 1, pad=0, sharex=ax)
    text_transform = ax.get_xaxis_transform()

    ax.plot(bin_mids, counts, "-", lw=3, color=bg_blue)
    ax_d.plot(bin_mids, deriv, "-", lw=3, color=bg_bluegrey)
    for peak_x, peak_y in peak_coords:
        # ax.plot(peak_x, peak_y, "x", ms=10, color=bg_red)
        ax.axvline(peak_x, lw=2, linestyle="--", color=bg_grey)
        ax.text(
            peak_x+2000, 0.99, isotope,
            transform=text_transform,
            fontsize=fontsize,
            ha="left",
            va="top"
        )
    for peak_x, peak_y in deriv_coords:
        # ax_d.plot(peak_x, peak_y, "x", ms=10, color=bg_red)
        ax_d.axvline(peak_x, lw=2, linestyle="--", color=bg_grey)

    ax_d.set_xlabel("Pulse integral (ADC channels x1000)", fontsize=fontsize)
    ax.set_ylabel("Counts\nx1000", fontsize=fontsize)
    ax_d.set_ylabel("Derivative\nx1000", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax_d.tick_params(labelsize=fontsize)
    ax_d.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax_d.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")

In [ ]:
def plot_figure_17b_1(
    ax: mpl.axes.Axes
):
    plot_data = figure_data["b"]
    isotope_colors = {
        # "241AmBe": bg_blue,
        "22Na": bg_bluegrey,
        "60Co": bg_red,
        "137Cs": bg_green,
        "152Eu": "black",
        "40K": bg_blue
    }
    label_ys = {
        25500.0: 0.99,
        49500.0: 0.94,
        67500.0: 0.89,
        85500.0: 0.84,
        119500.0: 0.79,
        147500.0: 0.74,
        148500.0: 0.69,
        166500.0: 0.64,
        171500.0: 0.59,
    }
    for isotope_name, isotope_plot_data in plot_data.items():
        bins = isotope_plot_data["bins"]
        counts = isotope_plot_data["counts"]
        peak_coords = isotope_plot_data["peak_coords"]
        compton_edges = isotope_plot_data["compton_edges"]
        color = isotope_colors[isotope_name]

        bin_mids = (bins[1:] + bins[:-1]) / 2
        text_transform = ax.get_xaxis_transform()

        ax.plot(bin_mids, counts, "-", lw=3, color=color)
        for peak_x, peak_y, edge_l in zip(*peak_coords, compton_edges):
            label_y = label_ys[peak_x]
            # ax.plot(peak_x, peak_y, "x", ms=10, color=bg_red)
            ax.axvline(peak_x, 0, label_y, lw=2, linestyle="--", color=bg_grey)
            ax.text(
                peak_x+2000, label_y, f"{edge_l} MeVee",
                transform=text_transform,
                fontsize=fontsize-5,
                ha="left",
                va="top",
                color=color
            )

        ax.set_xlim(0, 250000)
        ax.set_xlabel("Pulse integral (ADC channels x1000)", fontsize=fontsize)
        ax.set_ylabel("Counts x1000", fontsize=fontsize)
        ax.tick_params(labelsize=fontsize)
        ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
        ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")

In [ ]:
def plot_figure_17b_2(
    ax: mpl.axes.Axes
):
    isotope_colors = {
        # "241AmBe": bg_blue,
        "22Na": bg_bluegrey,
        "60Co": bg_red,
        "137Cs": bg_green,
        "152Eu": "black",
        "40K": bg_blue
    }
    
    plot_data = figure_data["c"]
    calib_curve_data = plot_data["calib_curve_data"]
    fit_result = plot_data["fit_result"]

    isotopes, calib_adc, calib_l = zip(*calib_curve_data)
    colors = [isotope_colors[isotope_name] for isotope_name in isotopes]
    fit_x = np.linspace(0, 1.5, 100)
    fit_y = fit_result.slope * fit_x + fit_result.intercept

    ax.scatter(
        calib_l, calib_adc, 
        s=20,
        c=colors
    )
    ax.plot(
        fit_x, fit_y, "-",
        lw=3,
        color=bg_grey,
        alpha=0.5
    )

    # ax.set_xlim(0, 250000)
    ax.set_xlabel("Light output (MeVee)", fontsize=fontsize)
    ax.set_ylabel("Pulse integral\n(ADC channels x1000)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    # ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")

### Figure Creation

In [ ]:
fig, ax = plt.subplots(layout="constrained")
plot_figure_17a(ax, "60Co")

In [ ]:
fig, ax = plt.subplots(layout="constrained")
plot_figure_17a(ax, "137Cs")

In [ ]:
fig, ax = plt.subplots(layout="constrained")
plot_figure_17a(ax, "40K")

In [ ]:
fig, ax = plt.subplots(layout="constrained")
plot_figure_17b_1(ax)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
plot_figure_17b_2(ax)